# Where would a new brewery open, and what type would it be?

The snapshot has no opening dates, so a classic time-series forecast is not possible. What it does have is enough to
answer the question in three complementary ways and to test the answers honestly:

1. **Place and type probabilities.** For a new brewery: which country, which state inside it, which city inside the
   state, and which type. Small places borrow strength from their region, and a Good–Turing estimate gives the chance
   that it opens in a city that has no brewery yet.
2. **A density map of where exactly.** A kernel density of brewery positions on the sphere.
3. **The pipeline.** 634 records are breweries *in planning*: openings that are being prepared. They show where growth
   is concentrated and serve as an out-of-sample test of model 1, which never sees them.

**Assumption.** A new brewery is drawn from the same geography as today's operating breweries (planning and closed
records excluded). The planning test checks how far that assumption holds.

**Outputs** go to `data/prediction_results.db`; the dashboard's Prediction page reads the tuned settings and
validation tables from it. Runtime: about 5 minutes.

## 1. Setup

In [1]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

import prediction_core as pc
import quality_core as qc

pd.set_option("display.width", 200)
DATA = Path("data")
OUT_DB = DATA / "prediction_results.db"
ALPHAS = [0.05, 0.1, 0.25, 0.5, 1.0]          # additive smoothing of place shares
BETAS = [1, 5, 10, 20, 50, 100]                 # pseudo-records pulling a place's type mix toward its region
BANDWIDTHS_KM = [10, 20, 30, 40, 50, 60, 80]    # kernel density bandwidths
K = 5                                           # cross-validation folds

## 2. Data

Counts use **raw** (every source record, coordinates or not) with harmonized state labels; the density map uses the
geocoded **all_records** with the coordinate corrections applied.

In [2]:
with sqlite3.connect(DATA / "raw_breweries.db") as con:
    raw = qc.harmonize_states(qc.apply_corrections(pd.read_sql_query("SELECT * FROM raw_breweries", con)))
with sqlite3.connect(DATA / "analysis_all_records.db") as con:
    geo = qc.harmonize_states(qc.apply_corrections(pd.read_sql_query("SELECT * FROM analysis_all_records", con)))
op = pc.operating(raw)
geo_op = pc.operating(geo).dropna(subset=["latitude", "longitude"])
print(f"raw {len(raw):,} | operating {len(op):,} | planning {int((raw['brewery_type'] == 'planning').sum())} | "
      f"closed {int((raw['brewery_type'] == 'closed').sum())} | geocoded operating {len(geo_op):,}")

raw 11,959 | operating 10,680 | planning 634 | closed 645 | geocoded operating 8,820


## 3. The model

**Places.** For a parent place (the world, a country, a state) with children holding $n_1, \dots, n_K$ operating
breweries ($N = \sum n_k$):

$$P(\text{new place}) = \frac{N_1 + 1}{N + 1}, \qquad P(k) = \bigl(1 - P(\text{new place})\bigr)\,\frac{n_k + \alpha}{N + \alpha K}$$

where $N_1$ is the number of children holding exactly one brewery (Good–Turing).

**Types.** The type mix of a place is shrunk toward its parent's: with $m_t$ breweries of type $t$ in the place,
$m$ in total and $q_t$ the parent's mix, $P(t) = (m_t + \beta q_t)/(m + \beta)$, applied from the global mix down
to country, state and city.

**Place given type.** Bayes' rule: $P(\text{place} \mid t) \propto P(\text{place}) \cdot P(t \mid \text{place})$.

## 4. Tuning α and β by cross-validation

In [3]:
cv_places, cv_types = pc.cross_validate(op, ALPHAS, BETAS, k=K)
alpha = float(cv_places.groupby("alpha")["log_loss"].mean().idxmin())
beta = float(cv_types.loc[cv_types["log_loss"].idxmin(), "beta"])
print("chosen alpha:", alpha, "| chosen beta:", beta)
print(cv_places.round(3).to_string(index=False))
print(cv_types.round(3).to_string(index=False))

chosen alpha: 0.05 | chosen beta: 20.0
level  alpha  records  log_loss  top1_%  top3_%  log_loss_uniform
 city   0.05   2128.6     2.816  40.148  58.123             4.197
 city   0.10   2128.6     2.818  40.148  58.123             4.197
 city   0.25   2128.6     2.825  40.110  58.123             4.197
 city   0.50   2128.6     2.837  40.139  58.113             4.197
 city   1.00   2128.6     2.859  40.073  58.104             4.197
state   0.05   2135.8     2.945  21.313  38.431             3.414
state   0.10   2135.8     2.945  21.313  38.431             3.414
state   0.25   2135.8     2.945  21.313  38.412             3.414
state   0.50   2135.8     2.945  21.303  38.412             3.414
state   1.00   2135.8     2.945  21.285  38.393             3.414
 beta  records  log_loss  log_loss_state_only  log_loss_global  top1_%  top1_state_only_%  top1_global_%
    1   2136.0     0.896                0.827            1.028  66.049             68.352         55.393
    5   2136.0     0.821 

### How good is it?

Log-loss is the average of −log(probability given to what actually happened): lower is better, and the uniform
baseline shows what knowing nothing would score. Top-1 is how often the most probable option was the right one.

In [4]:
best_places = cv_places[cv_places["alpha"] == alpha].set_index("level")
best_types = cv_types[cv_types["beta"] == beta].iloc[0]
summary = pd.DataFrame([
    {"task": "state, given the country", "log_loss": best_places.loc["state", "log_loss"],
     "baseline": "uniform over the country's states", "baseline_log_loss": best_places.loc["state", "log_loss_uniform"],
     "top1_%": best_places.loc["state", "top1_%"], "top3_%": best_places.loc["state", "top3_%"]},
    {"task": "city, given the state", "log_loss": best_places.loc["city", "log_loss"],
     "baseline": "uniform over the state's cities", "baseline_log_loss": best_places.loc["city", "log_loss_uniform"],
     "top1_%": best_places.loc["city", "top1_%"], "top3_%": best_places.loc["city", "top3_%"]},
    {"task": "type, given the city", "log_loss": best_types["log_loss"], "baseline": "global type mix",
     "baseline_log_loss": best_types["log_loss_global"], "top1_%": best_types["top1_%"], "top3_%": np.nan},
    {"task": "type, given the state only", "log_loss": best_types["log_loss_state_only"],
     "baseline": "global type mix", "baseline_log_loss": best_types["log_loss_global"],
     "top1_%": best_types["top1_state_only_%"], "top3_%": np.nan}])
summary.round(3)

,task,log_loss,baseline,baseline_log_loss,top1_%,top3_%
0,"state, given the country",2.945,uniform over the country's states,3.414,21.313,38.431
1,"city, given the state",2.816,uniform over the state's cities,4.197,40.148,58.123
2,"type, given the city",0.801,global type mix,1.028,68.184,NaN
3,"type, given the state only",0.806,global type mix,1.028,68.408,NaN


## 5. The pipeline: planning breweries as future openings

Planning records exist only for the United States. First: is their geography different from that of operating
breweries? Then: how well does a model trained only on operating breweries predict where they are?

In [5]:
pipeline = pc.pipeline_by_state(raw)
ct = pipeline[(pipeline["operating"] > 0)][["operating", "planning"]]
chi2, p_value, dof, _ = chi2_contingency(ct.T.to_numpy())
cramers_v = float(np.sqrt(chi2 / (ct.to_numpy().sum() * (min(ct.shape) - 1))))
print(f"planning vs operating state mix: chi2 = {chi2:.1f}, dof = {dof}, p = {p_value:.2e}, Cramér's V = {cramers_v:.3f}")
ptest = pc.planning_test(raw, alpha)
print(json.dumps(ptest, indent=1))
pipeline.head(15).round(2)

planning vs operating state mix: chi2 = 186.0, dof = 50, p = 1.52e-17, Cramér's V = 0.153


{
 "state": {
  "level": "state",
  "alpha": 0.05,
  "records": 634,
  "log_loss": 3.5010528221578485,
  "top1_%": 10.094637223974763,
  "top3_%": 22.082018927444793,
  "log_loss_uniform": 3.951243718581426
 },
 "city": {
  "level": "city",
  "alpha": 0.05,
  "records": 634,
  "log_loss": 3.084287365563844,
  "top1_%": 35.01577287066246,
  "top3_%": 53.62776025236593,
  "log_loss_uniform": 4.567759272384226
 },
 "new_city": {
  "predicted_%": 29.480687118319864,
  "observed_%": 35.01577287066246,
  "records": 634
 }
}


,state_province_harmonized,operating,planning,closed,operating_share_%,planning_share_%,lift,planning_per_100_operating,closed_per_100_operating
4,California,804,64,51,11.05,10.09,0.91,7.96,6.34
43,Texas,294,56,2,4.04,8.83,2.19,19.05,0.68
5,Colorado,401,39,10,5.51,6.15,1.12,9.73,2.49
9,Florida,275,38,0,3.78,5.99,1.59,13.82,0.00
32,New York,382,37,0,5.25,5.84,1.11,9.69,0.00
38,Pennsylvania,309,36,0,4.25,5.68,1.34,11.65,0.00
46,Virginia,221,34,0,3.04,5.36,1.77,15.38,0.00
33,North Carolina,283,28,0,3.89,4.42,1.14,9.89,0.00
35,Ohio,276,26,1,3.79,4.10,1.08,9.42,0.36
13,Illinois,234,23,0,3.22,3.63,1.13,9.83,0.00


## 6. Density: bandwidth by held-out likelihood

In [6]:
kde_cv = pc.kde_cv(geo_op["latitude"].to_numpy(), geo_op["longitude"].to_numpy(), BANDWIDTHS_KM, k=K)
bandwidth = float(kde_cv.loc[kde_cv["heldout_log_likelihood"].idxmax(), "bandwidth_km"])
model = pc.kde_fit(geo_op["latitude"], geo_op["longitude"], bandwidth)
plan_geo = geo[geo["brewery_type"] == "planning"].dropna(subset=["latitude", "longitude"])
planning_ll = pc.kde_score(model, plan_geo["latitude"], plan_geo["longitude"])
print("chosen bandwidth (km):", bandwidth, "| mean log-density of the", len(plan_geo),
      "geocoded planning breweries:", round(planning_ll, 3))
kde_cv.round(3)

chosen bandwidth (km): 30.0 | mean log-density of the 108 geocoded planning breweries: 3.246


,bandwidth_km,heldout_log_likelihood
0,10,-0.701
1,20,2.011
2,30,2.293
3,40,2.281
4,50,2.210
5,60,2.130
6,80,1.979


## 7. Save

In [7]:
params = pd.DataFrame([{"key": "alpha", "value": alpha}, {"key": "beta", "value": beta},
                       {"key": "bandwidth_km", "value": bandwidth}, {"key": "folds", "value": K},
                       {"key": "planning_chi2", "value": chi2}, {"key": "planning_p_value", "value": p_value},
                       {"key": "planning_cramers_v", "value": cramers_v},
                       {"key": "planning_kde_log_density", "value": planning_ll}])
ptest_tbl = pd.DataFrame([{"test": k, **v} for k, v in ptest.items()])
tables = {"cv_places": cv_places, "cv_types": cv_types, "summary": summary, "pipeline_by_state": pipeline,
          "planning_test": ptest_tbl, "kde_cv": kde_cv, "params": params}
with sqlite3.connect(OUT_DB) as con:
    for name, t in tables.items():
        t.to_sql(name, con, if_exists="replace", index=False)
print({k: len(v) for k, v in tables.items()})

{'cv_places': 10, 'cv_types': 6, 'summary': 4, 'pipeline_by_state': 51, 'planning_test': 3, 'kde_cv': 7, 'params': 8}


## 8. What the models say

In [8]:
m = pc.PlaceTypeModel(alpha, beta).fit(op)
print("Most likely countries:", m.place_distribution("country").head(5).round(3).to_dict())
print("Most likely US states:", m.place_distribution("state", ("United States",)).head(5).round(3).to_dict())
print("Type of a new brewery in the US:", m.type_distribution("United States").head(4).round(3).to_dict())
print("Top pipeline lift (states with ≥ 10 planning):",
      pipeline[pipeline["planning"] >= 10].nlargest(5, "lift")[["state_province_harmonized", "lift"]].round(2).values.tolist())

Most likely countries: {'United States': 0.681, 'Germany': 0.124, 'Australia': 0.048, 'Belgium': 0.043, 'Canada': 0.025}
Most likely US states: {'California': 0.11, 'Colorado': 0.055, 'New York': 0.052, 'Washington': 0.05, 'Michigan': 0.049}
Type of a new brewery in the US: {'micro': 0.579, 'brewpub': 0.335, 'regional': 0.029, 'contract': 0.026}
Top pipeline lift (states with ≥ 10 planning): [['Georgia', 2.69], ['Tennessee', 2.22], ['Texas', 2.19], ['Connecticut', 1.83], ['Virginia', 1.77]]


### Reading the results

**Where a brewery opens decides its type, at the state level.** Knowing the state cuts the type log-loss from 1.03
(global mix) to 0.81 and lifts top-1 accuracy from 55% to 68%. Knowing the city adds almost nothing on top (0.801 vs
0.806): each country's and state's labelling habits (Germany's brewpubs, Australia's and Belgium's micro breweries)
carry the signal, not the town.

**Place predictions beat knowing nothing, by a wide margin at city level.** State given country: log-loss 2.95
against 3.41 for a uniform guess; city given state: 2.82 against 4.20. For the city, the single most likely answer is
often "a city with no brewery yet", which is why top-1 accuracy (40%) is mostly the model correctly expecting a new
town.

**The pipeline points south and east.** Planning breweries are not spread like operating ones (χ² p = 1.5 × 10⁻¹⁷,
Cramér's V 0.15, a modest but clear shift). Relative to their share of today's breweries, Georgia carries 2.7× more of
the openings in preparation, Tennessee 2.2×, Texas 2.2×, Connecticut 1.8× and Virginia 1.8×; Michigan (0.7×) and
California (0.9×) carry fewer. A model trained only on operating breweries still beats a uniform guess on these future
openings (state 3.50 vs 3.95, city 3.08 vs 4.57) but does worse than in its own cross-validation (2.95): that gap is
the cost of the geography moving.

**New towns.** Good–Turing expects 29.5% of new breweries to open in a city that has none yet; 35.0% of the planning
breweries do. Expansion into new towns is slightly faster than the current map implies.

**Where exactly.** The best kernel bandwidth is 30 km, narrowly ahead of 40 km (held-out log-likelihood 2.29 against
2.28). The density is summed exactly over every brewery (`prediction_core.HaversineKDE`): scikit-learn's ball-tree
version, used at first, was badly off far from the data at small bandwidths and had overrated 10 and 20 km. The
geocoded planning breweries sit in denser places than a typical brewery (mean log-density 3.25 against 2.29 held
out), so openings cluster where breweries already are — with
a strong caveat: only 108 of the 634 planning records have coordinates and 64 of those are in California, so this
comparison mostly describes California.

**What this is and is not.** These are probabilities for the next brewery under the assumption that it resembles
today's operating breweries, corrected by the only forward-looking signal the data holds (planning records, US
only). They are not a forecast of how many breweries will open or when.
